In [ ]:
!pip install transformers datasets evaluate -q


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
import numpy as np

import evaluate

c:\Users\prasa\OneDrive\Desktop\BE\backend\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
ds = load_dataset("google-research-datasets/go_emotions")
train_ds = ds["train"]
val_ds = ds["validation"]
test_ds = ds["test"]

In [1]:
import pandas as pd

In [2]:
df = pd.read_parquet("../data/text/train-00000-of-00001.parquet")

In [3]:
print(df.columns.tolist())

['text', 'labels', 'id']


In [11]:
df.head(10)

,text,labels,id
0,My favourite food is anything I didn't have to...,[27],eebbqej
1,"Now if he does off himself, everyone will thin...",[27],ed00q6i
2,WHY THE FUCK IS BAYLESS ISOING,[2],eezlygj
3,To make her feel threatened,[14],ed7ypvh
4,Dirty Southern Wankers,[3],ed0bdzj
5,OmG pEyToN iSn'T gOoD eNoUgH tO hElP uS iN tHe...,[26],edvnz26
6,Yes I heard abt the f bombs! That has to be wh...,[15],ee3b6wu
7,We need more boards and to create a bit more s...,"[8, 20]",ef4qmod
8,Damn youtube and outrage drama is super lucrat...,[0],ed8wbdn
9,It might be linked to the trust factor of your...,[27],eczgv1o


In [10]:
print(df["label"].value_counts())

KeyError: 'label'

In [12]:
for i in range(6):
    print("\nLABEL =", i)
    print(df[df["label"] == i].head(3))


LABEL = 0


KeyError: 'label'

In [4]:
id2label = {
    0:"admiration",1:"amusement",2:"anger",3:"annoyance",4:"approval",
    5:"caring",6:"confusion",7:"curiosity",8:"desire",9:"disappointment",
    10:"disapproval",11:"disgust",12:"embarrassment",13:"excitement",
    14:"fear",15:"gratitude",16:"grief",17:"joy",18:"love",19:"nervousness",
    20:"optimism",21:"pride",22:"realization",23:"relief",24:"remorse",
    25:"sadness",26:"surprise",27:"neutral"
}

In [5]:
final_map = {
    "admiration":0, "amusement":0, "approval":0, "caring":0, "desire":0,
    "excitement":0, "gratitude":0, "joy":0, "love":0, "optimism":0,
    "pride":0, "relief":0,

    "curiosity":1, "realization":1, "surprise":1, "neutral":1,

    "anger":2, "annoyance":2, "disapproval":2, "confusion":2,

    "fear":3, "nervousness":3,

    "disappointment":4, "disgust":4, "embarrassment":4, "remorse":4,

    "grief":5, "sadness":5
}

In [6]:
def convert_label(example):
    labels = example["labels"]
    
    if len(labels) == 0:
        example["label"] = 1
    else:
        first = labels[0]
        emotion = id2label[first]
        example["label"] = final_map[emotion]
        
    return example

train_ds = train_ds.map(convert_label)
val_ds = val_ds.map(convert_label)
test_ds = test_ds.map(convert_label)

In [7]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [8]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

In [9]:
keep = ["input_ids","attention_mask","label"]

for dsx in [train_ds, val_ds, test_ds]:
    remove_cols = [c for c in dsx.column_names if c not in keep]
    dsx = dsx.remove_columns(remove_cols)

train_ds = train_ds.remove_columns([c for c in train_ds.column_names if c not in keep])
val_ds = val_ds.remove_columns([c for c in val_ds.column_names if c not in keep])
test_ds = test_ds.remove_columns([c for c in test_ds.column_names if c not in keep])

train_ds.set_format("torch")
val_ds.set_format("torch")
test_ds.set_format("torch")

In [10]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=6
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 4530.71it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
acc = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": acc.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels, average="weighted")["f1"]
    }

In [ ]:
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir="../models/text_final_6",

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=5,

    weight_decay=0.01,
    warmup_ratio=0.1,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,

    logging_steps=100,

    fp16=True,   # True only if CUDA GPU available
    report_to="none"
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [13]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

In [14]:
trainer.train()

c:\Users\prasa\OneDrive\Desktop\BE\backend\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.760457,0.786440,0.700700,0.701242
2,0.673758,0.777359,0.708441,0.705423
3,0.550110,0.824369,0.709362,0.706380
4,0.388568,1.142395,0.693144,0.690313
5,0.324489,1.406309,0.687984,0.687206


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.36it/s]
c:\Users\prasa\OneDrive\Desktop\BE\backend\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.02it/s]
c:\Users\prasa\OneDrive\Desktop\BE\backend\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.96it/s]
c:\Users\prasa\OneDrive\Desktop\BE\backend\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00

TrainOutput(global_step=27135, training_loss=0.5626127443723045, metrics={'train_runtime': 46719.1048, 'train_samples_per_second': 4.646, 'train_steps_per_second': 0.581, 'total_flos': 7188524971545600.0, 'train_loss': 0.5626127443723045, 'epoch': 5.0})

In [9]:
trainer.evaluate(test_ds)

NameError: name 'trainer' is not defined

In [18]:
model.save_pretrained("../models/text_final_6")
tokenizer.save_pretrained("../models/text_final_6")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.39it/s]


('../models/text_final_6\\tokenizer_config.json',
 '../models/text_final_6\\tokenizer.json')

In [8]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average="weighted")
recall = recall_score(y_true, y_pred, average="weighted")
f1 = f1_score(y_true, y_pred, average="weighted")

print("Accuracy :", round(accuracy,4))
print("Precision:", round(precision,4))
print("Recall   :", round(recall,4))
print("F1 Score :", round(f1,4))

NameError: name 'y_true' is not defined

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [7]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained("../models/text_final_6")
print(config.id2label)
print(config.label2id)

{0: 'LABEL_0', 1: 'LABEL_1', 2: 'LABEL_2', 3: 'LABEL_3', 4: 'LABEL_4', 5: 'LABEL_5'}
{'LABEL_0': 0, 'LABEL_1': 1, 'LABEL_2': 2, 'LABEL_3': 3, 'LABEL_4': 4, 'LABEL_5': 5}
